In [37]:
from dotenv import load_dotenv
load_dotenv()

True

In [38]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter


from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


from langchain_groq import ChatGroq


from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate


from langchain.agents import create_agent

In [39]:
loader = PyPDFLoader("IItm.pdf") 
docs = loader.load()
len(docs)

72

In [40]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200 )
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

225

In [41]:

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs, 
    embedding=embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3903.23it/s]


In [42]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about IITM BS Degree.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    
    return context

In [43]:
llm  = ChatGroq(model="openai/gpt-oss-20b")

In [44]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [45]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [46]:
query = "What is the credits requirement for Bsc, and what are the courses in Bsc"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  IITM BS Degree credits requirement Bsc courses
Tool Called:  IITM BS Degree BSc credit requirement 114 credits courses


In [47]:
result = response["messages"][-1].content

In [48]:
print(result)

**IITM BS (B.Sc.) – Credit Requirement**

| Item | Detail |
|------|--------|
| **Total credits needed for the degree** | **114 credits** (as per the university’s prescribed norms) |
| **Credit structure** | • 24 credits can be earned from on‑campus courses at IIT Madras.  <br>• 24 credits can be earned from non‑graded courses (apprenticeship, other institutes, NPTEL, etc.).  <br>• 28 credits (16 Theory + 12 Apprenticeship) are allowed for the MS track.  <br>• 3 campus credits (only from IIT Madras) equal 1 BS credit.  <br>• Students may take a maximum of 4 courses from the same institute (travel to campus required).  <br>• Credits earned from campus courses count toward the CGPA, while other campus courses are credit‑transfer only. |

> **Bottom line:** Once a student has accumulated **114 credits** (including the above combinations), they are eligible to receive the B.Sc. degree and can proceed to the BS level.

---

**Courses in the B.Sc. Programme**

The B.Sc. curriculum is built a